# 08 Stage 2 Clustering — Woman-Level Risk Archetypes

**Owner:** PBC  
**Depends on:** `07_data_integration.ipynb` (or `scripts/run_stage2_data_prep.py`)  
**Outputs:** `cluster_assignments.csv`, `cluster_profiles.csv`, `cluster_k_selection.csv`, K-Means model + scaler

In [ ]:
import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path('..').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.clustering.kmeans_cluster import (
    CLUSTER_FEATURES,
    build_cluster_profiles,
    fit_clusters,
)

STAGE2_DIR = PROJECT_ROOT / 'data' / 'processed' / 'stage2'
OUTPUTS_DIR = PROJECT_ROOT / 'outputs' / 'stage2_results'
MODELS_DIR = PROJECT_ROOT / 'saved_models' / 'stage2'

In [ ]:
# Confirm Stage 2 preclustering matrix exists (includes OOF barrier probabilities)
precluster_path = STAGE2_DIR / 'X_stage2_preclustering.csv'
if not precluster_path.exists():
    print('Preclustering matrix missing — run scripts/run_stage2_data_prep.py first.')
else:
    X_pre = pd.read_csv(precluster_path)
    print(f'Loaded X_stage2_preclustering: {X_pre.shape}')
    missing = [c for c in CLUSTER_FEATURES if c not in X_pre.columns]
    print('Clustering features present:', not missing)
    if missing:
        print('Missing:', missing)

In [ ]:
# PBC Quick-start #1: confirm m14 and v626a in raw extract
raw_df = pd.read_csv(PROJECT_ROOT / 'data' / 'raw' / 'NFHS5_Individual.csv', low_memory=False)
raw_df.columns = raw_df.columns.str.strip()
print('m14 present:', 'm14' in raw_df.columns)
print('v626a present:', 'v626a' in raw_df.columns)
if 'v626a' in raw_df.columns:
    print(raw_df['v626a'].value_counts(dropna=False).head(10))

In [ ]:
# Fit MiniBatchKMeans on full 724K matrix; silhouette scored on 20K subsample
cluster_labels, model, scaler, k_meta = fit_clusters(X_pre, models_dir=MODELS_DIR)

OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)
cluster_labels.to_csv(OUTPUTS_DIR / 'cluster_assignments.csv', index=False)
k_meta.to_csv(OUTPUTS_DIR / 'cluster_k_selection.csv', index=False)

chosen_k = int(k_meta.loc[k_meta['chosen'], 'k'].iloc[0])
print(f'Chosen k: {chosen_k}')
print(f'Cluster distribution:\n{cluster_labels.value_counts().sort_index()}')

In [ ]:
# Build named archetype profiles (Guide Section 9, tasks 5–6)
profiles = build_cluster_profiles(X_pre, cluster_labels, outputs_dir=OUTPUTS_DIR)
display_cols = ['cluster', 'n_women', 'archetype_name', 'vulnerability_score', 'composite_barrier_score']
profiles[display_cols]